# 02 - Preprocessing: from waveforms to spectrograms and scalograms

For every record:
1. the mean of the recorded samples is removed from each component (zero padding stays zero);
2. the analysis window is placed around an anchor time (`window.anchor`, by default the maximum of the energy
   envelope of the three components), from `pre_anchor_s` before to `post_anchor_s` after; the same window is used
   for the three components;
3. the window, extended by a context margin on both sides, is tapered, low-pass filtered and resampled;
4. spectrogram (STFT) and scalogram (Morlet CWT) are computed on the extended window and cropped to the window,
   on a common grid of logarithmic frequencies and time bins;
5. the power is converted to absolute decibels and stored as arrays of shape (components, frequencies, times).

Every parameter is read from the `preprocessing` section of `configs/default.yaml`.
The records come from `waveforms.source` (development subset now, full dataset later).

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.inspection import share_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.sampling import sample_records_per_label
from wqu.data.waveforms import (
    iter_waveform_batches,
    load_waveforms,
    select_inspection_records,
    stored_sampling_rate_hz,
)
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    pooled_power_values,
    preprocess_and_store,
    process_record,
    read_features,
    window_table,
)
from wqu.preprocessing.anchoring import (
    anchor_disagreement,
    anchor_table,
    energy_envelope,
    energy_kept_by_method,
    missing_anchor_share,
)
from wqu.preprocessing.transform import power_to_db
from wqu.preprocessing.window_search import (
    collect_window_metrics,
    padding_reference,
    padding_scores,
    pareto_front,
    select_window,
    summarize_window_grid,
    window_row,
)
from wqu.preprocessing.windowing import prepare_trace
from wqu.visualization import (
    apply_plot_style,
    plot_anchor_comparison,
    plot_anchor_windows,
    plot_representation_comparison,
    plot_value_distributions,
    plot_window_grid_heatmap,
    plot_window_overlay,
    plot_window_tradeoff,
    save_figure,
    save_figures_pdf,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
apply_plot_style()
columns = config.columns
preprocessing = config.preprocessing
tolerance = config.waveforms.zero_tolerance
label_names = {subset.label: subset.name for subset in config.data.subsets}
for directory in (preprocessing.figures_dir, preprocessing.tables_dir, preprocessing.figures_dir / "png_checks"):
    directory.mkdir(parents=True, exist_ok=True)

grid = grid_from_config(preprocessing)
print(f"Window: {preprocessing.pre_anchor_s:g} s before to {preprocessing.post_anchor_s:g} s after the anchor "
      f"({preprocessing.anchor_method}) "
      f"(+ {preprocessing.context_margin_s:g} s margin on each side)")
print(f"Resampling: {preprocessing.target_sampling_rate_hz:g} Hz after a low-pass at {preprocessing.antialias_corner_hz:g} Hz")
print(f"Grid: {preprocessing.n_frequencies} frequencies from {preprocessing.min_frequency_hz:g} to "
      f"{preprocessing.max_frequency_hz:g} Hz (log), {preprocessing.n_times} time bins of "
      f"{preprocessing.window_length_s / preprocessing.n_times:.3f} s")
print(f"STFT window: {preprocessing.stft_window_s:g} s ({preprocessing.stft_window_type}); "
      f"Morlet w0 = {preprocessing.morlet_w0:g}")

In [ ]:
_, metadata = load_esm25_metadata(config)
records = select_inspection_records(metadata, config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
print(f"Waveform source: {config.waveforms.source}; records: {len(records):,}")
records[columns.label].value_counts().rename(index=label_names)

## 1. Analysis window: what is kept and what is cut

For every record:
- `anchor_time_s`: time of the anchor from the start of the stored trace;
- `nonzero_cut_before_c`, `nonzero_cut_after_c`: recorded (non-zero) samples of component c removed before and after the window;
- `missing_in_window_s_c`: seconds of the window without data on component c (window reaching beyond the recording);
- `recorded_fraction_kept`, `energy_fraction_kept`: share of the recorded samples and of the signal energy (sum of squared amplitudes) inside the window.

In [ ]:
windows = window_table(iter_waveform_batches(records, config), sampling_rate_hz, preprocessing, tolerance)
windows = records[[columns.label, "trace_name_old", columns.station_id]].join(windows)
for side in ("before", "after"):
    windows[f"cut_{side}_s"] = windows[[f"nonzero_cut_{side}_{c}" for c in CV_COMPONENTS]].max(axis=1) / sampling_rate_hz
windows["missing_in_window_s"] = windows[[f"missing_in_window_s_{c}" for c in CV_COMPONENTS]].max(axis=1)
windows.to_csv(preprocessing.tables_dir / f"window_statistics_{config.waveforms.source}.csv")

WINDOW_COLUMNS = ["anchor_time_s", "cut_before_s", "cut_after_s", "missing_in_window_s", "recorded_fraction_kept", "energy_fraction_kept"]
numeric_summary_by_label(windows, WINDOW_COLUMNS, columns.label).rename(index=label_names, level=columns.label).round(3)

In [ ]:
window_flags = pd.DataFrame(
    {
        "window starts before the recording": windows["window_start"] < 0,
        "window ends after the stored trace": windows["window_stop"] > config.data.waveform_length_s * sampling_rate_hz,
        "some data missing in the window": windows["missing_in_window_s"] > 0,
        "energy kept < 95%": windows["energy_fraction_kept"] < 0.95,
        "energy kept < 80%": windows["energy_fraction_kept"] < 0.80,
    }
)
share_by_label(window_flags, windows[columns.label]).rename(columns=label_names).round(3)

## 2. Padding leakage and choice of the window

The zero padding of the stored traces differs between the classes (notebook 01: the trailing padding is longer
in good records). If a window reaches into the padding, the amount of padding it contains can reveal the class
without looking at the waveform. For every candidate window from `window_search.pre_anchor_s` before to
`window_search.post_anchor_s` after the anchor (plus the configured window) two properties are measured:

- **leakage**: the class separation max(AUC, 1 - AUC) of the padding, where 0.5 means no information, measured
  (i) on the seconds of padding before the anchor, after the anchor and in total (worst component), one at a time, and
  (ii) with a classifier that sees only the padding (seconds per component and fraction of padding in
  `leakage.classifier.mask_bins` time bins per component), evaluated out of fold with folds grouped by event.
  The classifier tests whether the padding differs between the classes in any way, not only in amount.
  A window is accepted when all separations are at most `window_search.leakage.max_separation`.
  Records with a component entirely without data are excluded from these tests when
  `leakage.exclude_empty_components` is true: their padding is a real defect, not an effect of the window;
- **signal loss**: fraction of the energy of the stored trace kept in the window, on the records of
  `window_search.energy.reference_label`, summarized by `window_search.energy.criterion`.

Among accepted windows the one with the highest energy score is suggested, choosing the shortest among scores
within `window_search.score_tolerance`. The suggestion is only printed: the preprocessing always uses
`window.pre_anchor_s` and `window.post_anchor_s` of the configuration, which must be updated by hand.

The separation is measured on the records of this notebook (ESM25, used for training and validation only),
so the choice of the window does not use test data.

In [ ]:
search = preprocessing.window_search
window_metrics = collect_window_metrics(
    iter_waveform_batches(records, config),
    sampling_rate_hz,
    pre_anchor_s=(*search.pre_anchor_s, preprocessing.pre_anchor_s),
    post_anchor_s=(*search.post_anchor_s, preprocessing.post_anchor_s),
    tolerance=tolerance,
    label_column=columns.label,
    group_column=columns.event_id,
    anchor_method=preprocessing.anchor_method,
    anchoring=preprocessing.anchoring,
)
print(f"Records with a component without data: {int(window_metrics.geometry.has_empty_component.sum())} "
      f"({'excluded from' if search.exclude_empty_components else 'included in'} the leakage tests)")
window_summary = summarize_window_grid(window_metrics, search)
window_summary["pareto"] = pareto_front(window_summary)
window_summary.to_csv(preprocessing.tables_dir / f"window_search_{config.waveforms.source}.csv", index=False)

configured = window_row(window_summary, preprocessing.pre_anchor_s, preprocessing.post_anchor_s)
print(f"Configured window ({preprocessing.pre_anchor_s:g} s, {preprocessing.post_anchor_s:g} s): "
      f"padding separation {configured['padding_separation']:.3f} "
      f"(threshold {search.max_padding_separation:g}; classifier {configured['separation_padding_classifier']:.3f}), energy score {configured['energy_score']:.3f}, "
      f"{'accepted' if configured['accepted'] else 'rejected'}")
configured.to_frame("configured window")

In [ ]:
suggested = select_window(window_summary, search)
print(f"Suggested window: pre_anchor_s = {suggested['pre_anchor_s']:g}, post_anchor_s = {suggested['post_anchor_s']:g} "
      f"(length {suggested['window_length_s']:g} s), padding separation {suggested['padding_separation']:.3f}, "
      f"energy score {suggested['energy_score']:.3f}")
print("To use it, set preprocessing.window.pre_anchor_s and post_anchor_s in the configuration and rerun the notebook.")

SUMMARY_COLUMNS = ["pre_anchor_s", "post_anchor_s", "window_length_s", "separation_padding_before_anchor_s",
                   "separation_padding_after_anchor_s", "separation_padding_total_s", "separation_padding_classifier",
                   "padding_separation",
                   "share_with_padding_0", "share_with_padding_1", "energy_score", "median_energy_fraction_kept",
                   "accepted"]
window_summary[window_summary["pareto"]].sort_values("padding_separation")[SUMMARY_COLUMNS].round(3)

In [ ]:
marked_windows = {
    "suggested": (suggested["pre_anchor_s"], suggested["post_anchor_s"]),
    "configured": (preprocessing.pre_anchor_s, preprocessing.post_anchor_s),
}
grid_summary = window_summary[window_summary["pre_anchor_s"].isin(search.pre_anchor_s)
                              & window_summary["post_anchor_s"].isin(search.post_anchor_s)]
figure, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
plot_window_grid_heatmap(grid_summary, "padding_separation", axes[0],
                         "padding separation, largest of all tests (hatched: rejected)",
                         "Oranges", ".3f", marked_windows)
plot_window_grid_heatmap(grid_summary, "energy_score", axes[1],
                         f"energy score ({search.energy_criterion}) on {label_names[search.energy_reference_label]}",
                         "Blues", ".3f", marked_windows)
save_figure(figure, preprocessing.figures_dir / f"window_search_grid_{config.waveforms.source}.png")
plt.show()

figure, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
plot_window_tradeoff(window_summary, window_summary["pareto"], search.max_padding_separation, marked_windows, ax)
save_figure(figure, preprocessing.figures_dir / f"window_search_tradeoff_{config.waveforms.source}.png")
plt.show()

### Padding reference of the configured window

The window is fixed (`window.pre_anchor_s`, `window.post_anchor_s`) and a small residual of class information in the
padding is accepted; it is measured and reported instead of being used to reject windows.
- **Reference table**: separation reached by the classifier that sees only the padding, compared with the separations
  reached after permuting the labels `leakage.classifier.n_permutations` times (event groups kept), i.e. what the
  classifier reaches by chance. A network trained on the images must clearly exceed this level for its performance
  not to be explained by the padding alone.
- **Per-record score**: out-of-fold probability of the bad class given only the padding, saved next to the features
  (`padding_scores.csv`), to be compared later with the predictions and uncertainty of the network.

In [ ]:
padding_reference_table = padding_reference(
    window_metrics, preprocessing.pre_anchor_s, preprocessing.post_anchor_s, preprocessing.anchor_method, search
)
padding_reference_table.to_csv(
    preprocessing.tables_dir / f"padding_reference_{config.waveforms.source}.csv", index=False
)

record_padding_scores = records[[columns.label, "trace_name_old", columns.event_id]].join(
    padding_scores(window_metrics, preprocessing.pre_anchor_s, preprocessing.post_anchor_s, search)
)
padding_scores_path = feature_file_path(preprocessing, config.waveforms.source).with_name("padding_scores.csv")
padding_scores_path.parent.mkdir(parents=True, exist_ok=True)
record_padding_scores.to_csv(padding_scores_path)
print(f"Per-record padding scores saved in {padding_scores_path}")
print(record_padding_scores.groupby(columns.label)["padding_score"].describe().rename(index=label_names).round(3))
padding_reference_table.round(3)

## 3. Comparison of anchoring methods

The method used by the preprocessing is `window.anchor`. The PGA can fall on an isolated spike or on another event
recorded in the same trace; the methods compared (see `wqu.preprocessing.anchoring` and `preprocessing.anchoring`):
- `envelope`: maximum of the energy envelope (squared amplitudes of the three components averaged over `envelope_window_s`);
- `arias`: time at which `arias_fraction` of the energy (Arias intensity summed over the components) has arrived;
- `theoretical`: catalogue origin time + distance / `phase_velocity_km_s`;
- `hybrid`: envelope maximum within `hybrid_search_half_width_s` of the theoretical arrival (catalogued event, precise timing).

For each method, per class: share of records without an anchor (a difference between the classes would itself leak
the label), pairwise disagreement (anchors more than `agreement_tolerance_s` apart) and energy kept by the configured
window placed on the anchor.

In [ ]:
anchoring = preprocessing.anchoring
anchors = anchor_table(
    iter_waveform_batches(records, config), sampling_rate_hz, anchoring,
    preprocessing.pre_anchor_s, preprocessing.post_anchor_s, tolerance,
)
anchors = records[[columns.label, "trace_name_old"]].join(anchors)
anchors.to_csv(preprocessing.tables_dir / f"anchors_{config.waveforms.source}.csv")
anchor_labels = anchors[columns.label]
missing_anchor_share(anchors, anchor_labels, anchoring.methods).rename(columns=label_names).round(3)

In [ ]:
anchor_disagreement(anchors, anchor_labels, anchoring.methods, anchoring.agreement_tolerance_s).unstack(
    "label"
).rename(columns=label_names, level="label").round(3)

In [ ]:
ENERGY_QUANTILES = [0.05, 0.1, 0.25, 0.5]
energy_kept_by_method(anchors, anchor_labels, anchoring.methods, ENERGY_QUANTILES).rename(
    index=label_names, level="label"
).round(3)

Records where the configured anchor and the PGA disagree, for each class (dashed: PGA).
The bottom panel shows the energy envelope used by the `envelope` and `hybrid` methods.

In [ ]:
configured_method = preprocessing.anchor_method
if configured_method != "pga" and {"pga", configured_method} <= set(anchoring.methods):
    disagreement_s = (anchors["anchor_s_pga"] - anchors[f"anchor_s_{configured_method}"]).abs()
    disagreeing = anchors[disagreement_s > anchoring.agreement_tolerance_s]
    anchor_examples = pd.concat(
        group.sample(n=min(anchoring.n_examples_per_class, len(group)), random_state=config.waveforms.random_seed)
        for _, group in disagreeing.groupby(columns.label)
    ) if len(disagreeing) else disagreeing
    anchor_traces = load_waveforms(records.loc[anchor_examples.index], config) if len(anchor_examples) else []
    for (index, row), trace in zip(anchor_examples.iterrows(), anchor_traces):
        prepared = prepare_trace(trace, tolerance)
        envelope = energy_envelope(prepared.centred, sampling_rate_hz, anchoring.envelope_window_s)
        figure = plot_anchor_comparison(
            prepared.centred, sampling_rate_hz,
            {method: row[f"anchor_s_{method}"] for method in anchoring.methods},
            envelope, CV_COMPONENTS,
            f"{label_names[row[columns.label]]}: {row['trace_name_old']}",
        )
        save_figure(figure, preprocessing.figures_dir / f"anchors_{row['trace_name_old']}.png")
        plt.show()
else:
    print("These examples need a configured anchor other than pga, and both methods in anchoring.methods.")

### Gallery of the windows of every method

Random records of each class (`anchoring.gallery_records_per_class`), chosen independently of the anchors.
Each page shows the three components and, below, the window that each method would produce with the configured
lengths (bar) and its anchor (black tick). One PDF per class in `figures_dir`; the first page of each class is also
shown here.

In [ ]:
def anchor_window_figures(selected: pd.DataFrame):
    """Yield one figure per selected record with the windows of every anchoring method."""
    traces = load_waveforms(records.loc[selected.index], config)
    for (index, row), trace in zip(selected.iterrows(), traces):
        yield plot_anchor_windows(
            prepare_trace(trace, tolerance).centred, sampling_rate_hz,
            {method: row[f"anchor_s_{method}"] for method in anchoring.methods},
            preprocessing.pre_anchor_s, preprocessing.post_anchor_s, CV_COMPONENTS,
            f"{label_names[row[columns.label]]}: {row['trace_name_old']}",
        )


gallery = pd.concat(
    group.sample(n=min(anchoring.gallery_records_per_class, len(group)), random_state=config.waveforms.random_seed)
    for _, group in anchors.groupby(columns.label)
)
for label, selected in gallery.groupby(columns.label):
    first = next(anchor_window_figures(selected.iloc[:1]))
    plt.show()
    plt.close(first)
    gallery_path = save_figures_pdf(
        anchor_window_figures(selected),
        preprocessing.figures_dir / f"anchor_gallery_{label_names[label]}_{config.waveforms.source}.pdf",
    )
    print(f"{label_names[label]}: {len(selected)} records in {gallery_path}")

## 4. Visual check of the cut

The full stored trace in light grey and the analysis window in dark grey (shaded):
random records of each class, then, for each class, the records that keep the smallest share of their energy.

In [ ]:
N_RANDOM_EXAMPLES = 2
N_LOWEST_ENERGY_EXAMPLES = 2

random_examples = windows.groupby(columns.label, group_keys=False).sample(
    n=N_RANDOM_EXAMPLES, random_state=config.waveforms.random_seed
)
lowest_energy = pd.concat(
    group.nsmallest(N_LOWEST_ENERGY_EXAMPLES, "energy_fraction_kept") for _, group in windows.groupby(columns.label)
)
examples = pd.concat([random_examples, lowest_energy])
example_traces = load_waveforms(records.loc[examples.index], config)

for (index, window), trace in zip(examples.iterrows(), example_traces):
    title = (f"{label_names[window[columns.label]]}: {window['trace_name_old']} - "
             f"energy kept {window['energy_fraction_kept']:.1%}")
    figure = plot_window_overlay(trace, sampling_rate_hz, int(window["window_start"]), int(window["window_stop"]),
                                 CV_COMPONENTS, title)
    save_figure(figure, preprocessing.figures_dir / f"window_{window['trace_name_old']}.png")
    plt.show()

## 5. Spectrogram and scalogram of one record

Comparison figure for a record of the station with most records in the inspected set
(waveform of the window, spectrogram and scalogram; one row per component).

In [ ]:
example_station = records[columns.station_id].value_counts().index[0]
station_record = records[records[columns.station_id] == example_station].iloc[[0]]
station_trace = load_waveforms(station_record, config)[0]
processed = process_record(station_trace, station_record.iloc[0], sampling_rate_hz, preprocessing, grid, tolerance)
station_db = {name: power_to_db(processed.power[name], preprocessing.reference_power, preprocessing.floor_power)
              for name in REPRESENTATIONS}

record_name = station_record["trace_name_old"].iloc[0]
figure = plot_representation_comparison(
    processed.window_waveform, preprocessing.target_sampling_rate_hz,
    station_db["spectrogram"], station_db["scalogram"],
    grid.frequency_edges_hz, grid.times_s, CV_COMPONENTS,
    f"{label_names[station_record[columns.label].iloc[0]]}: {record_name}",
)
save_figure(figure, preprocessing.figures_dir / f"comparison_{example_station}.png")
print({name: values.shape for name, values in station_db.items()})

## 6. Power distributions and decibel floor

Power values of a sample of records, pooled over components, frequencies and times.
Zeros (components without data, windows reaching beyond the recording) cannot be converted to decibels:
they are raised to `decibel.floor_power`. The floor should lie below the weakest real values (instrument noise),
so that it marks missing data without clipping real signal. On the development subset real values lie above about
-120 dB and filter artefacts below, so the floor is set to 1e-12 (-120 dB).

In [ ]:
distribution_records = sample_records_per_label(
    records,
    label_column=columns.label,
    group_column=columns.event_id,
    records_per_label=preprocessing.n_distribution_records // 2,
    max_records_per_group=None,
    seed=config.waveforms.random_seed,
)
pooled = pooled_power_values(
    iter_waveform_batches(distribution_records, config), sampling_rate_hz, preprocessing, tolerance
)
QUANTILES = [0.0, 0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0]
summary = pd.DataFrame({name: np.quantile(values, QUANTILES) for name, values in pooled.items()}, index=QUANTILES)
summary.index.name = "quantile"
for name, values in pooled.items():
    print(f"{name}: zeros {np.mean(values == 0):.4%}, below the current floor "
          f"({preprocessing.floor_power:g}) {np.mean(values < preprocessing.floor_power):.4%}")
summary

In [ ]:
figure, ax = plt.subplots(figsize=(10, 4))
positive_db = {name: 10.0 * np.log10(values[values > 0] / preprocessing.reference_power) for name, values in pooled.items()}
floor_db = 10.0 * np.log10(preprocessing.floor_power / preprocessing.reference_power)
highest_db = max(float(values.max()) for values in positive_db.values())
plot_value_distributions(positive_db, ax, "power [dB re reference_power] (zeros excluded)",
                         x_limits=(floor_db - preprocessing.distribution_plot_margin_db, highest_db))
ax.axvline(floor_db, color="black", linewidth=1.0, linestyle="--")
ax.set_title(f"dashed line: floor_power; x range from {preprocessing.distribution_plot_margin_db:g} dB below it",
             loc="left")
figure.tight_layout()
save_figure(figure, preprocessing.figures_dir / f"power_distributions_{config.waveforms.source}.png")

## 7. Compute and store all records

One HDF5 file per waveform source in `preprocessing.output_dir`, with the decibel arrays of both representations
(float16, not yet standardized: the standardization statistics will be computed on the training set only),
the grid and the preprocessing parameters; `records.csv` next to it links each row (`feature_index`) to its record.

In [ ]:
features_path = feature_file_path(preprocessing, config.waveforms.source)
stored_records = preprocess_and_store(
    records, lambda selected: iter_waveform_batches(selected, config), sampling_rate_hz, preprocessing, tolerance,
    features_path,
)
print(f"Stored {len(stored_records):,} records in {features_path}")

## 8. PNG images for visual inspection

Comparison figures read back from the stored arrays (spectrogram and scalogram) for a few records of each class,
saved in `figures_dir/png_checks/`. These images are for inspection only; the networks use the stored arrays.

In [ ]:
png_records = stored_records.groupby(columns.label, group_keys=False).sample(
    n=preprocessing.n_png_records_per_class, random_state=config.waveforms.random_seed
)
rows = png_records[FEATURE_INDEX_COLUMN].to_numpy()
stored_db = {name: read_features(features_path, name, rows)[0] for name in REPRESENTATIONS}
png_traces = load_waveforms(records.loc[png_records.index], config)

for position, ((index, record), trace) in enumerate(zip(png_records.iterrows(), png_traces)):
    window_waveform = process_record(trace, record, sampling_rate_hz, preprocessing, grid, tolerance).window_waveform
    figure = plot_representation_comparison(
        window_waveform, preprocessing.target_sampling_rate_hz,
        stored_db["spectrogram"][position], stored_db["scalogram"][position],
        grid.frequency_edges_hz, grid.times_s, CV_COMPONENTS,
        f"{label_names[record[columns.label]]}: {record['trace_name_old']}",
    )
    save_figure(figure, preprocessing.figures_dir / "png_checks" / f"{label_names[record[columns.label]]}_{record['trace_name_old']}.png")
    plt.close(figure)
print(f"Saved {len(png_records)} PNG images in {preprocessing.figures_dir / 'png_checks'}")